## Phase 1: Load and split

### Imports

In [3]:
import sys
sys.path.append("..")          # lets the notebook see the src/ folder

import numpy as np
import pandas as pd
from src.metrics import evaluate, build_test_liked

LIKE_THRESHOLD = 4.0           # a rating of 4+ counts as "liked"
K= 10                          # recommend 10 movies per user

### Load

In [5]:
ratings = pd.read_csv("../data/ml-25m/ratings.csv",
                      dtype={"userId": "int32", "movieId": "int32",
                             "rating": "float32", "timestamp": "int64"})
movies = pd.read_csv("../data/ml-25m/movies.csv")

ratings["date"] = pd.to_datetime(ratings["timestamp"], unit="s")

print(ratings.shape, movies.shape)
print("date range:", ratings.date.min().date(), "to", ratings.date.max().date())
movies.head()

(25000095, 5) (62423, 3)
date range: 1995-01-09 to 2019-11-21


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
ratings.head()

,userId,movieId,rating,timestamp,date
0,1,296,5.0,1147880044,2006-05-17 15:34:04
1,1,306,3.5,1147868817,2006-05-17 12:26:57
2,1,307,5.0,1147868828,2006-05-17 12:27:08
3,1,665,5.0,1147878820,2006-05-17 15:13:40
4,1,899,3.5,1147868510,2006-05-17 12:21:50


### Filter

In [8]:
# A recommender needs enough signal per user and per item to learn from.
# Keep users with >= 20 ratings and movies with >= 50.
u_counts = ratings.userId.value_counts()
m_counts = ratings.movieId.value_counts()

r = ratings[ratings.userId.isin(u_counts[u_counts >= 20].index) & ratings.movieId.isin(m_counts[m_counts >= 50].index)]

print(f"kept {len(r):,} of {len(ratings):,} ratings")
print(f"{r.userId.nunique():,} users, {r.movieId.nunique():,} movies")

kept 24,644,928 of 25,000,095 ratings
162,540 users, 13,176 movies


In [9]:
r.head()

,userId,movieId,rating,timestamp,date
0,1,296,5.0,1147880044,2006-05-17 15:34:04
1,1,306,3.5,1147868817,2006-05-17 12:26:57
2,1,307,5.0,1147868828,2006-05-17 12:27:08
3,1,665,5.0,1147878820,2006-05-17 15:13:40
4,1,899,3.5,1147868510,2006-05-17 12:21:50


In [10]:
u_counts.head()

userId
72315     32202
80974      9178
137293     8913
33844      7919
20055      7488
Name: count, dtype: int64

In [11]:
m_counts.head()

movieId
356     81491
318     81482
296     79672
593     74127
2571    72674
Name: count, dtype: int64

### Time-based split

In [13]:
# Time-based split: train on everything before the cutoff, test on what comes after.
CUTOFF = "2018-01-01"

train = r[r.date <  CUTOFF]
test  = r[r.date >= CUTOFF]

# Only evaluate on users we've actually seen before, and items that exist in train.
known_users = set(train.userId.unique())
known_items = set(train.movieId.unique())
test = test[test.userId.isin(known_users) & test.movieId.isin(known_items)]

print(f"train: {len(train):,} ratings, {train.userId.nunique():,} users")
print(f"test : {len(test):,} ratings, {test.userId.nunique():,} users")

train: 22,261,221 ratings, 150,543 users
test : 414,622 ratings, 6,460 users


### What each test user liked

In [15]:
test_liked = build_test_liked(test, LIKE_THRESHOLD)

print(f"{len(test_liked):,} evaluable users")
print(f"median liked items per user: {test_liked.apply(len).median():.0f}")

6,269 evaluable users
median liked items per user: 13


In [16]:
test_liked.sample(10)

userId
29144     {4103, 106889, 89492, 21, 2710, 2968, 177689, ...
100193    {72641, 115713, 6565, 104841, 68237, 1262, 155...
21161     {31658, 1580, 60397, 164909, 6287, 8368, 54001...
70213     {172577, 55269, 60040, 30793, 160874, 56587, 5...
81403     {115713, 515, 54286, 56339, 103444, 3098, 3100...
155503    {122912, 115713, 166528, 179817, 178061, 17761...
4835                                 {152017, 2542, 151199}
15920     {111360, 97921, 137857, 48516, 89864, 48394, 4...
141039    {2662, 4102, 6535, 26662, 31658, 119145, 1663,...
146710                                               {1251}
Name: movieId, dtype: object

---
## Phase 2: The evaluation harness

### Sanity-check the harness before trusting it

In [19]:
# A random recommender should score near zero. If not, the harness is broken.
all_items = train.movieId.unique()
rng = np.random.default_rng(42)

def random_rec(user_id, k):
    return list(rng.choice(all_items, size=k, replace=False))

print(evaluate(random_rec, test_liked, k=K, n_users=500).to_string())

recall       0.0020
precision    0.0026
ndcg         0.0028


### Save for notebook 02

In [21]:
cols = ["userId", "movieId", "rating"]
train[cols].to_parquet("../data/train.parquet", index=False)
test[cols].to_parquet("../data/test.parquet", index=False)
print("saved")

saved
